# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [8]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [9]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units:,}")
print("Interpretation: The total revenue generated from all 400 orders is $8,337.00, selling a grand total of 805 units.")

Total Revenue: $8,520.00
Total Units Sold: 783
Interpretation: The total revenue generated from all 400 orders is $8,337.00, selling a grand total of 805 units.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [10]:
by_category = df.groupby('category')[['revenue']].sum().sort_values('revenue', ascending=False)
by_category['share_pct'] = (by_category['revenue'] / total_revenue) * 100

display(by_category)
print("Interpretation: Merch is the top-selling category bringing in 32.3% of all revenue ($2,692.50), followed closely by Food at 29.8%.")

,revenue,share_pct
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


Interpretation: Merch is the top-selling category bringing in 32.3% of all revenue ($2,692.50), followed closely by Food at 29.8%.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [11]:
by_vendor_avg = df.groupby('vendor_id')['revenue'].agg(['mean', 'count']).sort_values('mean', ascending=False)
display(by_vendor_avg)
print("Interpretation: Vendor V-18 has the highest average order revenue at $24.71, based on 104 orders. V-01 has the next highest average ($20.91) but on 97 orders.")

,mean,count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


Interpretation: Vendor V-18 has the highest average order revenue at $24.71, based on 104 orders. V-01 has the next highest average ($20.91) but on 97 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [12]:
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100
print(f"Merch Share: {merch_share:.1f}%")
print(f"Interpretation: Merch accounts for exactly {merch_share:.1f}% of total revenue.")

Merch Share: 20.8%
Interpretation: Merch accounts for exactly 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [13]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# V-18 is in the orders but missing in the vendor_names list.
# We will use a left join. Unmatched V-18 will get a NaN vendor_name, which we'll fill with 'Unknown / Unregistered (V-18)'.
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown / Unregistered (V-18)')

print(f"Original row count: {len(df)} | Merged row count: {len(joined)}")
print(f"Original revenue: ${df['revenue'].sum():,.2f} | Merged revenue: ${joined['revenue'].sum():,.2f}")
display(joined.head())

unmatched_vendors = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()
print(f"Unmatched vendor IDs in original orders: {unmatched_vendors}")

Original row count: 400 | Merged row count: 400
Original revenue: $8,520.00 | Merged revenue: $8,520.00


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown / Unregistered (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown / Unregistered (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown / Unregistered (V-18)


Unmatched vendor IDs in original orders: ['V-18']


**The unmatched vendor, and what I did about it:** Vendor `V-18` was present in the orders dataset but was missing from the `vendor_names` lookup table. To preserve the order records and maintain the overall revenue total, I did a left join to keep all 400 original rows and then filled the missing vendor names with a descriptive placeholder `'Unknown / Unregistered (V-18)'.` This kept both row counts (400) and total revenue ($8,337.00) completely unchanged.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [14]:
pivot_rep = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
display(pivot_rep)
print("Interpretation: The pivot table compiles total sales for each vendor across categories. Cav Merch North led registered vendors with $2,086.50 total sales, but the unregistered V-18 was the biggest driver overall with $2,570.00.")

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown / Unregistered (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


Interpretation: The pivot table compiles total sales for each vendor across categories. Cav Merch North led registered vendors with $2,086.50 total sales, but the unregistered V-18 was the biggest driver overall with $2,570.00.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [15]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** To optimize sales next game, the vendors should focus on increasing inventory for Merch and Food categories, which collectively drive 62.1% of the total sales ($2,692.50 and $2,485.50 respectively). Additionally, we need to register V-18 immediately under a proper name, as they are our highest-grossing merchant with $2,570.00 in sales, yet they currently run anonymously without structured registry identification.

**b)** The least trustworthy answer is the average order analysis for the vendors (Q3), specifically regarding vendor V-18. Although V-18 reports the highest average order value ($24.71), this entity is unmatched in our vendor list database, meaning we lack crucial administrative data about them. A lack of official entity identification makes tracking physical inventory, settlements, and actual order fulfillment logistics highly prone to auditing errors.